# 2HRX9P6HKXA8V

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import os
import re
import tabulate
from IPython.display import display, Markdown

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter

Preemptively set new Pandas option, also set matplotlib to close

In [ ]:
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

Allow reloading of custom Python classes without resetting kernel

In [ ]:
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_merged
%store -r sales_data_merged
%store -r restaurants_by_4m_coverage
%store -r time_differences
%store -r time_differences_details

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

Time Differences

In [ ]:
%store -r restaurant_data_unprocessed
timezones_acronyms = {}
for loc_id, df in restaurant_data_unprocessed.items():
    time = df['created_at'].iloc[0]
    timezone = time.strip('0123456789-+: ')
    timezones_acronyms[loc_id] = timezone
timezones = {
    '0RJH3FFPYBPEY': 'America/New_York',
    '1SQPTEGYPH0GA': 'America/Denver',
    '3AXDVZJYN9DRS': 'Europe/London',
    '75WYSXR9QBK5M': 'Pacific/Honolulu',
    '78AY09MVJVTYE': 'America/New_York',
    '9XKJD8DQTH559': 'America/New_York',
    'AQD04SM0J92WA': 'America/Los_Angeles',
    'CB2KHY1C2G9PT': 'America/New_York',
    'EMBVNVD207CC6': 'America/New_York',
    'JHDN7CF1C03X5': 'America/Chicago',
    'L3XS7WSJ4AJA3': 'Europe/London',
    'L69HYJ4Y3TR91': 'America/New_York',
    'LBMCPAYT7W36V': 'America/New_York',
    'LBZEEFSBJNB3Z': 'America/Los_Angeles',
    'LFZFT3VASXPED': 'Australia/Sydney',
    'LQ5EH4BKGV61T': 'America/New_York',
    'LZ5MR1TS37E7W': 'America/Los_Angeles',
    'MS8R16DY0JQAM': 'America/Los_Angeles',
    'N0PC58FB2XAZ3': 'America/Chicago',
    'S8MT0YGD2KTN9': 'America/New_York',
    'SAFK7ND1HR6XS': 'America/Los_Angeles',
    'SRQS8F7JWA9MZ': 'America/New_York',
    'V3Q26BHF3SE2H': 'America/New_York',
    'W8T41JZK0ZMEP': 'America/New_York',
    'WJA3YCD4QBWRX': 'America/New_York',
    '1G5AJ17XCH2A8': 'America/Chicago',
    'ADPFRN3QZRCXK': 'America/Los_Angeles',
    'ED5J990H5VAZT': 'America/Los_Angeles',
    '2HRX9P6HKXA8V': 'America/Los_Angeles',
    'C0BE4NDSW26QN': 'America/New_York'
}
for loc_id, df in sales_and_menu_data.items():
    df.index = df.index.tz_convert(timezones[loc_id])

before_after_details.loc[restaurants_by_4m_coverage,'first_plant_based_mention'] = before_after_details.loc[restaurants_by_4m_coverage,'first_plant_based_mention'].str.title()

In [ ]:
loc_id = '2HRX9P6HKXA8V'
df = sales_and_menu_data[loc_id]

In [ ]:
sales_and_menu_data[loc_id].index

In [ ]:
time_differences_details[loc_id]

10 Hour Difference

In [ ]:
sales_and_menu_data[restaurants_by_4m_coverage[0]]['item_name'].value_counts().sort_values(ascending=False).head(10)

In [ ]:
sales_and_menu_data[restaurants_by_4m_coverage[0]].loc[pd.Timestamp('2019-07-17 4:45:58+00:00'):pd.Timestamp('2019-07-18').tz_localize('UTC')].head(5)

20 Hour Difference

In [ ]:
sales_and_menu_data[restaurants_by_4m_coverage[0]].loc[pd.Timestamp('2020-06-07 22:27:20+00:00'):pd.Timestamp('2020-06-09').tz_localize('UTC')].head(10)

In [ ]:
time_differences['2HRX9P6HKXA8V']

In [ ]:
time_differences_details['2HRX9P6HKXA8V'][0][time_differences_details['2HRX9P6HKXA8V'][0] == 5]

In [ ]:
sales_and_menu_data['2HRX9P6HKXA8V'].loc['2020-08-24 4:55:15+00:00':'2020-08-25 10:20:15+00:00']

In [ ]:
dish_names = {
    "Big Bob Bratwurst": ["Big Bob"],
    "Warm Bavarian Pretzel": ["Bavarian Pretzel", "Pretzel"],
    "Hans Jalapeno & Cheddar": ["Han's Jalapeno & Cheddar", "Jalapeno & Cheddar", "Jalapeño & Cheddar", "Hans' Jalape√±O & Cheddar", "Jalape√±O & Cheddar"],
    "Dirtyface Beer Wurst": ["Beer Wurst"],
    "Spinach Organic Chicken": [], # "Chicken", "Organic Chicken"
    "Italian Organic Chicken" : [],
    "Organic Chicken Sausage" : [],
    "Helgas Giant Kelbassi": ["Helga's Giant Kelbassi", "Kelbassi", "Giant Kelbassi", "Helga'S Giant Kelbassi"],
    "Large Sauerkraut - 8Oz Bowl": ["Side Sauerkraut - 8Oz Bowl", "Side Saurkraut"],
    #"Veggie Wurst": ["Vegetarian"],
    "Tims Cascade Potato Chips": ["Tim's Cascade Potato Chips", "Tim'S Cascade Potato Chips"], # "Kettle Brand Potato Chips", "Potato Chips", "Chips"
    "Gluhwein": ["Glühwein"],
    "Turkey Dog": ["Organic Turkey Dog"],
    "Soup": [], #"Veggie Soup" "Vegan Soup", "House Vegan Soup"
    #"Chili": ["Vegan Chili", "Chili Con Carne"],
    "Gingerbread Cookie": ["Haus Made Gingerbread Cookie"],
    #"Egift Card": ["Gift Card", "Promotional $5 Gift Certificates", "Donation $5 Gift Certificates"],
    #"Vegan Chili": ["Vegan Soup"],
    #"Weisswurst": ["Bockwurst"],
    #"Beyond Sausage": ["Veggie Wurst", "Vegetarian"],
    
    "Bottled Water": ["Athena Bottled Water"],
    "Pepsi Bottled Sodas": ["Pepsi Fountain", "Diet Pepsi Fountain", "Pepsi", "Diet Pepsi", "Pepsi Fountain Sodas", "Pepsi Bottled Sodas 20Oz"],
    "Dr. Pepper Fountain": ["Dr. Pepper"],
    "7-Up Fountain": ["7-Up", "-Up Fountain", "-Up"],
    "Mountain Dew Fountain": ["Mountain Dew"],
    "Rootbeer Fountain": ["Rootbeer"],
    
}

alcohol_related_items = {
    "Icicle Premium Pilsner": [],
    "Dirtyface Amber Lager": ["Dirtyface Beer Wurst", "Dirtyface Amber Mustard", "To-Go Dirtyface Amber Single 16Oz Can", "To-Go Dirtyface Amber 22Oz Bottle", "To-Go Dirtyface Amber 4 Pack 16Oz Cans", "Bottled Dirtyface"],
    "Bootjack IPA": ["Bootjack Ipa", "To-Go Bootjack Ipa Single Can 12Oz", "To-Go Bootjack Ipa 6 Pack 12Oz"],
    "Alpenhaze Hazy IPA": ["Alpenhaze"],
    "Colchuck Raspberry Wheat": ["To-Go Colchuck Raspberry Wheat Single Can 16Oz", "To-Go Colchuck Raspberry Wheat 4 Pack 16Oz", "Raspberry Dark Persuasion"],
    "Dark Persuasion": ["Dark Persuasion Chocolate Cake Ale", "To-Go Dark Persuasion German Chocolate Cake Ale Single Can 12Oz", "To-Go Dark Persuasion German Chocolate Cake Ale 6 Pack 12Oz"],
    "Hofbräu Original": ["Hofbr√§U Original"],
    "Hofbräu Hefe Weizen": ["Hofbrau Hefeweizen", "Hofbr√§U Hefeweizen", "Drubru Hefeweizen"],
    "Hofbräu Dunkel": ["Hofbr√§U Dunkel", "Hofbr√§U Dunkle", "Hofbrau Dunkel"],
    "Yonder Vantage Semi-Sweet Cider": ["Trailbreaker Cider 12Oz Can", "To-Go Trailbreaker Cider 12Oz Can", "Trailbreaker Cider 12Oz Can - Dine-In", "To-Go Trailbreaker Cider 12Oz Can *Takeout Only*", "Pitcher Draft Cider"],
    "Quartet Bordeaux-Style Blend": ["Cellars Trio", "Cellars Quartet", "Cellars Trio Bottle"],
    "Montage": ["Eagle Creek Montage (Merlot)", "Eagle Creek Montage", "Eagle Creek Montage Bottle"],
    "Chardonnay": ["Milbrandt Chardonnay Bottle"],
    "Pinot Grigio": ["Eagle Creek Pinot Grigio Bottle"],
    "Riesling": ["Ryan Patrick Riesling Bottle"],
    "Gewürztraminer": ["Icicle Ridge Gewurztraminer Bottle"],
    "Rosé of Sangiovese": ["Kestrel Ros√© Bottle", "Maryhill Ros√©", "Maryhill Ros√© Bottle"],
    "Ghostfish Brewing Company": ["Ghostfish Gf Can 12Oz"],
    "Athletic Brewing IPA": ["Athletic Brewing Ipa *Non-Alcoholic* 12Oz Can", "To-Go Athletic Brewing Ipa *Non-Alcoholic* 12Oz Can"],
    "Athletic Golden Ale": ["Athletic Brewing Blonde Ale *Non-Alcoholic & Gluten Free* 12Oz Can", "To-Go Athletic Brewing Blonde Ale *Non-Alcoholic & Gluten Free* 12Oz Can"],
    "Bitburger Drive Pilsner": ["N/A Beer - Bitburger"],
    "Crosscut Pilsner": ["To-Go Crosscut Pilsner Single Can 16Oz", "To-Go Crosscut Pilsner 4 Pack 16Oz"],
    "Kickstand Citra Pale Ale": ["Kickstand Pale Ale", "To-Go Kickstand Pale Ale Single Can 12Oz", "To-Go Kickstand Pale Ale 6 Pack 12Oz"],
    "Timbertown Brown": [],
    "Snow Creek K√∂Lsch": [],
    "Leavenworth Festbier": [],
    "Pamm'S American Lager": [],
    "Knock Off Australian Lager": [],
    "Enchantments Hazy Ipa": ["To-Go Enchantments Hazy Ipa Single Can 16Oz", "To-Go Enchantments Hazy Ipa Single Can 12Oz", "To-Go Enchantments Hazy Ipa 4 Pack 16Oz", "To-Go Enchantments Hazy Ipa 6 Pack 12Oz"],
    "One In Eight Fresh Hop Ipa": [],
    "Drumfire Dark Lager": [],
    "Drubru Kolsch": ["Drubru K√∂Lsch"],
    "Icicle Lager": [],
    "Gluten Free Beer": ["Gluten Free 16Oz - Dine-In", "To-Go Gluten Free 16Oz"],
    "N/A Beer": [],
    "Drubru Hefeweizen": ["Dru Bru K√∂Lsch"],
    "Sawdog IPA": ["Sawdog"],
    "Ryan Patrick Riesling Bottle": ["Riesling"],
}

non_alcoholic_drinks = [
    "Lemonade",
    "Pepsi Bottled Sodas",
    "Bottled Water",
    "Iced Tea",
    "Hot Cocoa",
    "Dr. Pepper Fountain",
    "Hot Tea",
    "Rootbeer Fountain",
    "7-Up Fountain",
    "Apple Juice",
    "Coffee",
    "Mountain Dew Fountain",
    "Pepsi Fountain Sodas 22Oz",
    "Gatorade Fountain",
    "Bottled Soda",
    "Gatorade",
    "Common Ground Coffee Amber",
    "Tap Water To-Go",
    "Fountain Refill"
]

merch = ["Souvenir Water Bottle"]

alcohols = list(alcohol_related_items.keys()) + ["Ibc 4 Pack Cans 16Oz", "Ibc 6 Pack Cans", "Ibc 6 Pack Cans 12Oz", "To-Go Single Cans Ibc 16Oz", "To-Go Single Cans Ibc 12Oz"]

others = {}

# Swap the keys and values
replacement_dict = {variant: canonical for canonical, variants in dish_names.items() for variant in variants}
alcohol_replacement_dict = {variant: canonical for canonical, variants in alcohol_related_items.items() for variant in variants}

# Items to remove
items_to_remove = []

In [ ]:
df_cleaned = (df
              .assign(item_name=lambda df: df['item_name']
                      .str.strip('123456789./\\ ')  # Clean up item names
                      .replace(replacement_dict)    # Replace names based on dictionary
                      .replace(alcohol_replacement_dict)
                      .replace(items_to_remove, pd.NA))  # Replace non-dish items with NA
              .dropna(subset=['item_name'])
              .assign(dish_category = lambda df: df['dish_category'].mask(df['item_name'].isin(alcohols), 'Alcohol'))
              .assign(dish_category = lambda df: df['dish_category'].mask(df['item_name'].isin(non_alcoholic_drinks), 'Drink'))
              .assign(dish_category = lambda df: df['dish_category'].mask(df['item_name'].isin(merch), 'Merch'))
              #.drop('unique_id', axis=1)
             )

df_cleaned.query('item_name in @alcohols').assign(dish_category = "Alcohol")

df = df_cleaned

In [ ]:
print(df.query('~dish_category.isin(["Alcohol", "Drink", "Merch"]) and is_plant_based == "Yes"')['item_name'].value_counts().to_string())

In [ ]:
print(df.query('~dish_category.isin(["Alcohol", "Drink", "Merch"]) and is_plant_based == "No"')['item_name'].value_counts().to_string())

In [ ]:
df.groupby('item_name')['dish_category'].unique()

In [ ]:
df.query('item_name == "Tims Cascade Potato Chips"')['item_modifications']

In [ ]:
df.query('item_name.str.contains("Kettle")')['item_modifications']

In [ ]:
df.query('item_name == "Potato Chips"')['item_modifications']